# **Data importing and data standardization and correlation heat map**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import copy


train = pd.read_excel("BNT-Energy2f.xlsx")
X=train.iloc[:,:-1]
Y=train.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)


scalerX = StandardScaler().fit(X_train)
scalery = StandardScaler().fit(y_train.values.reshape(-1,1))
X_train1 = scalerX.transform(X_train)
y_train1 = scalery.transform(y_train.values.reshape(-1,1))
X_test1 = scalerX.transform(X_test)
y_test1 = scalery.transform(y_test.values.reshape(-1,1))
y_new_inverse = scalery.inverse_transform(y_test1.reshape(-1,1))
X_train2=copy.deepcopy(X_train1)
y_train2=copy.deepcopy(y_train1)


from sklearn.linear_model import ElasticNet, Lasso, LinearRegression
from xgboost import XGBRegressor
# from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

xgb_model=XGBRegressor()

search_space={'n_estimators':[100,200,300,400,500,600,800],'max_depth':[3,4,6,7,9],'gamma0':[0.01],'learning_rate':[0.001,0.01,0.05,0.1,0.5,1]}


from sklearn.model_selection import GridSearchCV

GS=GridSearchCV(estimator=xgb_model,param_grid=search_space,scoring=["r2","neg_root_mean_squared_error"],refit="neg_root_mean_squared_error",cv=5,verbose=4)
GS.fit(X_train1,y_train1)

print(GS.best_params_) # ND gamma0=0.01, learning_rate=0.01, max_depth=9, n_estimators=800;
print(GS.best_score_)
from sklearn.metrics import r2_score
model_xgb=XGBRegressor(n_estimators=600,max_depth=4,eta=0.1)
model_xgb.fit(X_train1,y_train1)
#y_pred=model_xgb.predict(X_test1)

#y_new_inverse = scalery.inverse_transform(y_pred.reshape(-1,1))
#Y_true=scalery.inverse_transform(y_test1)
#r2_score(Y_true,y_new_inverse)

# **Model defining and training and best parameters with performance metric**


# **Model Prediction and plot with error bars using bootstrapping**

In [ ]:
Li = pd.read_excel("RSD.xlsx")
X_test=Li.iloc[:,:-1]
Y_test=Li.iloc[:,-1]

X_test1 = scalerX.transform(X_test)
Y_test1 = scalery.transform(Y_test.values.reshape(-1,1))


from sklearn.metrics import mean_squared_error

bootstrap_means=[]


N=500

for i in range(0,N):
  sample_index=np.random.choice(range(0,len(y_train1)),len(y_train1))

  X_samples=X_train1[sample_index]
  y_samples=y_train1[sample_index]
  print(i)

  model_xgb=XGBRegressor(n_estimators=800,max_depth=3,eta=0.01) #change parameters according to the best fit you achieved at the top!
  model_xgb.fit(X_samples, y_samples)
  y_prediction=model_xgb.predict(X_test1)


  bootstrap_means.append(scalery.inverse_transform(y_prediction.reshape(-1,1)))


bootstrap_means_array=np.array(bootstrap_means)


# Define file names for training and testing data
test_file = 'test_data_output_final.xlsx'

# Save the testing data to an Excel file
test_data_output_final = pd.concat([X_test, Y_test], axis=1)
test_data_output_final.to_excel(test_file, index=False)

# Add predictions to the 'test' DataFrame
test_data_output_final['Predicted_W'] = bootstrap_means_array.mean(0)
test_data_output_final['Error'] = bootstrap_means_array.std(0)

# Save the updated DataFrame with predictions to a new file
output_file = 'test_predicted_W.xlsx.'  # Update with the desired output file path
test_data_output_final.to_excel('test_predicted_W-newfeature-IE.xlsx', index=False)


print(mean_squared_error(scalery.inverse_transform(Y_test1),bootstrap_means_array.mean(0)))
print(np.sqrt(mean_squared_error(scalery.inverse_transform(Y_test1),bootstrap_means_array.mean(0))))



